In [1]:
# Base packages
import numpy as np
import matplotlib.pyplot as plt
import os
import copy

# I/O
import glob
from skimage import io
from PIL import Image
import h5py
import json
import pickle

# Pytorch
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor

# Data Augmentation
import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage
ia.seed(2)

# Plots
import seaborn as sns
import pandas as pd
from textwrap import wrap
import matplotlib.patheffects as path_effects
import statsmodels.api as sm

# Measurements and Metrics
from sklearn import metrics as skmetrics
from skimage import measure

from scipy.stats import pearsonr

from datetime import datetime

random_state = np.random.RandomState(42)

from network import make_network
import tqdm
import torch
import time
import nms
from dataset.data_loader import make_data_loader
from train.model_utils.utils import load_network
from evaluator.make_evaluator import make_evaluator
import argparse
import importlib
import glob
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import cv2
import matplotlib.patches as patches

/home/jiangb/.conda/envs/e2ec/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [3]:
def file_list_dict(img_list, gt_list):
    # Returns a dictionary with the list of images and corresponding ground truth
    file_list = {"imgs": img_list,
      "gt": gt_list}
    
    return file_list

In [4]:
class GVHD_Dataset(Dataset):
    def __init__(self, files, mode, transform=None):
        self.transform = transform 
        self.files = files
        self.mode = mode
        
    def __len__(self):
        size=len(self.files["imgs"])
        return size
    
    def __getitem__(self, idx):
        path_img = self.files["imgs"][idx]
        path_mask = self.files["gt"][idx]
        
        if self.mode == 'test':
            image = path_img
            mask = path_mask
        else:
            image = cv2.imread(path_img)
            mask = cv2.imread(path_mask)[:,:,0]
        
        return image, mask

In [5]:
def patch_extractor(img, patch_size = 256, stride=32):
    width = img.shape[1]
    height = img.shape[0]
    patch_list = []
    
    if len(img.shape) == 3:
        for j in range(0, width-patch_size+stride, stride):
            for i in range(0, height-patch_size+stride, stride):
                patch_list.extend([img[i:i+patch_size, j:j+patch_size, :]])
    elif len(img.shape) == 2:
        for j in range(0, width-patch_size+stride, stride):
            for i in range(0, height-patch_size+stride, stride):
                patch_list.extend([img[i:i+patch_size, j:j+patch_size]])
    
    return patch_list

In [6]:
def lesion_count(binary_mask):
    mask_labelled = measure.label(binary_mask)
    lesion_number = len(np.unique(mask_labelled))-1
    
    return lesion_number

In [7]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

composed = {
    'image':transforms.Compose([ToTensor()]),
    'mask':transforms.Compose([ToTensor()]),
}

contour_save_root = r'final_E2EC_contour_40' + os.sep
os.makedirs(contour_save_root, exist_ok=True)

In [8]:
def get_cfg():
    cfg = importlib.import_module('configs.' + 'coco').config
    cfg.test.with_nms = True
    cfg.test.segm_or_bbox = 'segm'
    cfg.test.test_stage = 'final-dml'
    return cfg

def bgr_to_rgb(img):
    return img[:, :, [2, 1, 0]]

def unnormalize_img(img, mean, std):
    """
    img: [3, h, w]
    """
    img = img.detach().cpu().clone()
    img *= torch.tensor(std).view(3, 1, 1)
    img += torch.tensor(mean).view(3, 1, 1)
    min_v = torch.min(img)
    img = (img - min_v) / (torch.max(img) - min_v)
    return img

In [9]:
def get_3rd_point(a, b):
    direct = a - b
    return b + np.array([-direct[1], direct[0]], dtype=np.float32)


def get_dir(src_point, rot_rad):
    sn, cs = np.sin(rot_rad), np.cos(rot_rad)

    src_result = [0, 0]
    src_result[0] = src_point[0] * cs - src_point[1] * sn
    src_result[1] = src_point[0] * sn + src_point[1] * cs

    return src_result

In [10]:
def get_affine_transform(center,
                         scale,
                         rot,
                         output_size,
                         shift=np.array([0, 0], dtype=np.float32),
                         inv=0):
    if not isinstance(scale, np.ndarray) and not isinstance(scale, list):
        scale = np.array([scale, scale], dtype=np.float32)

    scale_tmp = scale
    src_w = scale_tmp[0]
    dst_w = output_size[0]
    dst_h = output_size[1]

    rot_rad = np.pi * rot / 180
    src_dir = get_dir([0, src_w * -0.5], rot_rad)
    dst_dir = np.array([0, dst_w * -0.5], np.float32)

    src = np.zeros((3, 2), dtype=np.float32)
    dst = np.zeros((3, 2), dtype=np.float32)
    src[0, :] = center + scale_tmp * shift
    src[1, :] = center + src_dir + scale_tmp * shift
    dst[0, :] = [dst_w * 0.5, dst_h * 0.5]
    dst[1, :] = np.array([dst_w * 0.5, dst_h * 0.5], np.float32) + dst_dir

    src[2:, :] = get_3rd_point(src[0, :], src[1, :])
    dst[2:, :] = get_3rd_point(dst[0, :], dst[1, :])

    if inv:
        trans = cv2.getAffineTransform(np.float32(dst), np.float32(src))
    else:
        trans = cv2.getAffineTransform(np.float32(src), np.float32(dst))
    return trans

In [11]:
def augment(img, mean, std):
    # height, width = img.shape[0], img.shape[1]
    # center = np.array([img.shape[1] / 2., img.shape[0] / 2.], dtype=np.float32)
    # scale = np.array([width, height])
    # x = 32
    # input_w = (int(width / 1.) | (x - 1)) + 1
    # input_h = (int(height / 1.) | (x - 1)) + 1
    # center = np.array([width // 2, height // 2])
    # trans_input = get_affine_transform(center, scale, 0, [input_w, input_h])
    # inp = cv2.warpAffine(img, trans_input, (input_w, input_h), flags=cv2.INTER_LINEAR)
    inp = (img.astype(np.float32) / 255.)
    inp = (inp - mean) / std
    inp = inp.transpose(2, 0, 1)

    return inp

In [12]:
def patched_inference_list_E2EC(img, model, patch_size = 128, stride=32, include_mask = True):
    cfg = get_cfg()
    pad_val = patch_size-stride
    img_pad = np.pad(img, pad_width=[(pad_val, pad_val),(pad_val, pad_val),(0, 0)], mode='constant')
    
    patch_list = patch_extractor(img_pad, patch_size, stride)
    
    files_test = file_list_dict(patch_list, patch_list)
    gvhd_data_test = GVHD_Dataset(files_test,'test',composed)
    
    bbox_list = []
    conf_list = []
    mask_list = []
    index_list = []
    edge_list = []
    num_patches_h = len(range(0, img_pad.shape[0]-patch_size+stride, stride))
    num_patches_w = len(range(0, img_pad.shape[1]-patch_size+stride, stride))
    for img_idx in range(len(patch_list)):
        h_index = img_idx % num_patches_h
        w_index = np.floor(img_idx / num_patches_h)
        image, _ = gvhd_data_test[img_idx]
        # print(augment(image, cfg.data.mean, cfg.data.std).shape)
        inp = torch.tensor(augment(image, cfg.data.mean, cfg.data.std)).unsqueeze(0).float().cuda()
        with torch.no_grad():
            output = model(inp, None)
        if cfg.test.with_nms:
            nms.post_process(output)
        ex = output['py']
        ex = ex[-1] if isinstance(ex, list) else ex
        ex = ex.detach().cpu().numpy()
        # ex /= 2
        detection = output['detection']
        confidence = detection[:, 2]

        if len(confidence) > 0:
            for i in range(len(confidence)):
                box = np.zeros(4)
                box[0] = np.min(ex[i, :, 0])
                box[2] = np.max(ex[i, :, 0])
                box[1] = np.min(ex[i, :, 1])
                box[3] = np.max(ex[i, :, 1])
                # print(patch_size)
                if (box[0] < 1) or (box[1] < 1) or (box[2] > patch_size-12) or (box[3] > patch_size-12):
                    # edge_list.append(1)
                    continue
                else:
                    edge_list.append(0)
                box[0] = box[0] + w_index*stride
                box[2] = box[2] + w_index*stride
                box[1] = box[1] + h_index*stride
                box[3] = box[3] + h_index*stride
                bbox_list.append(box)
                conf = confidence[i]
                conf = float(conf)
                conf_list.append(conf)
                if include_mask:
                    mask = np.zeros((image.shape[0], image.shape[1]))
                    contour = ex[i].astype(np.int32)
                    cv2.fillPoly(mask, [contour], 1)
                    mask_list.append(mask)
                    index_list.append((h_index, w_index))
    if include_mask:
        return bbox_list, conf_list, edge_list, mask_list, index_list
    else:
        return bbox_list, conf_list, edge_list

In [13]:
def custom_nms(bounding_boxes, confidence_score, conf_threshold, iou_threshold, edge_list):
    # If no bounding boxes, return empty list
    if len(bounding_boxes) == 0:
        return [], {}
    
    grouping_dict = {}

    # Bounding boxes
    boxes = np.array(bounding_boxes)

    # coordinates of bounding boxes
    start_x = boxes[:, 0]
    start_y = boxes[:, 1]
    end_x = boxes[:, 2]
    end_y = boxes[:, 3]

    # Confidence scores of bounding boxes
    score = np.array(confidence_score)

    # Picked bounding boxes
    picked_boxes = []
    picked_score = []
    indices = []

    # Compute areas of bounding boxes
    areas = (end_x - start_x + 1) * (end_y - start_y + 1)

    # Sort by confidence score of bounding boxes
    order = np.argsort(score)
    order = np.array([i for i in order if confidence_score[i] > conf_threshold])

    # Iterate bounding boxes
    while order.size > 0:
        if edge_list[order[-1]] == 1:
            edge_list[order[-1]] = 0
            np.insert(order, 0, order[-1])
            order = order[0:-1]
            continue
        
        # The index of largest confidence score
        index = order[-1]

        # Pick the bounding box with largest confidence score
        picked_boxes.append(bounding_boxes[index])
        picked_score.append(confidence_score[index])
        indices.append(index)

        # Compute ordinates of intersection-over-union(IOU)
        x1 = np.maximum(start_x[index], start_x[order[:-1]])
        x2 = np.minimum(end_x[index], end_x[order[:-1]])
        y1 = np.maximum(start_y[index], start_y[order[:-1]])
        y2 = np.minimum(end_y[index], end_y[order[:-1]])

        # Compute areas of intersection-over-union
        w = np.maximum(0.0, x2 - x1 + 1)
        h = np.maximum(0.0, y2 - y1 + 1)
        intersection = w * h

        # Compute the ratio between intersection and union
        ratio = intersection / (areas[index] + areas[order[:-1]] - intersection)

        left = np.where(ratio < iou_threshold)
        merge = np.where(ratio >= iou_threshold)
        
        grouping_dict[index] = order[merge]
        order = order[left]

    return indices, grouping_dict

In [14]:
def seg_contour_overlay(mask_list,
                        image,
                        line_size=4,
                        fig_size=(16,16),
                        thresh=0.5,
                        color_list = ['blue','lime','magenta','yellow','red'],
                        save_fig=False,
                        save_name='overlay.jpg',
                        show_fig=True,
                        title='Contour Overlay',
                        save_fig_dpi=150,
                        contour_alpha=0.75
                       ):
    
    if torch.is_tensor(image):
        image = np.moveaxis(image.squeeze().cpu().numpy(), 0, -1)
    
    contour_list = []
    for mask in mask_list:
        contour_list.extend([measure.find_contours(mask, thresh)])
    
    fig, ax = plt.subplots(figsize=fig_size)
    image_new = image.copy()
    image[:,:,0] = image_new[:,:,2]
    image[:,:,2] = image_new[:,:,0]
    ax.imshow(image, cmap=plt.cm.gray, interpolation=None)
    
    color_select = 0
    for mask_contour in contour_list:
        for contour in mask_contour:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=line_size, color=color_list[color_select],alpha=contour_alpha)
        color_select+=1

    ax.set_xticks([])
    ax.set_yticks([])
    plt.title(title, fontsize = 24)
    if save_fig:
        plt.savefig(save_name, dpi=save_fig_dpi, bbox_inches='tight', pad_inches = 0)
    if show_fig:
        plt.show()
    else:
        plt.close()

In [15]:
def lesion_matching_visual(final_indices, gt_mask, mask_list, index_list, grouping_dict, current_photo_name, current_img):
    tp = 0
    fp = 0
    fn = 0
    
    patch_size = 256
    stride = 64
    pad_val = patch_size-stride

    padded_gt = np.pad(gt_mask, pad_width=[(pad_val, pad_val),(pad_val, pad_val)], mode='constant')
    gt_mask_labelled = measure.label(padded_gt)
    
    tp_map = np.zeros(padded_gt.shape)
    fp_map = np.zeros(padded_gt.shape)
    fn_map = np.zeros(padded_gt.shape)

    matched_gt_list = []
    for mask_index in final_indices:
        matched_gt = -1
        iou_val = 0
        pr_mask_full = np.zeros(padded_gt.shape)
        pr_mask_full[int(index_list[mask_index][0]*stride):int(index_list[mask_index][0]*stride+patch_size), int(index_list[mask_index][1]*stride):int(index_list[mask_index][1]*stride+patch_size)] =  mask_list[mask_index]
        pr_mask_full = pr_mask_full > 0.5
        
        for region_label in range (1,len(np.unique(gt_mask_labelled))):
            if region_label not in matched_gt_list:
                current_region_mask = np.zeros(padded_gt.shape)
                current_region_mask[gt_mask_labelled == region_label] = 1
                curr_iou = np.sum(pr_mask_full[current_region_mask == 1])/(np.sum(pr_mask_full) + np.sum(gt_mask_labelled == region_label) - np.sum(pr_mask_full[current_region_mask == 1]))
                if curr_iou > 0:
                    if curr_iou > iou_val:
                        iou_val = curr_iou
                        matched_gt = region_label
        if matched_gt != -1:
            tp += 1
            tp_map = np.maximum(pr_mask_full, tp_map)
            matched_gt_list.append(matched_gt)
        else:
            fp += 1
            fp_map = np.maximum(pr_mask_full, fp_map)
    
    fn = len(np.unique(gt_mask_labelled)) - len(matched_gt_list)
    
    for i in range (1,len(np.unique(gt_mask_labelled))):
        if i not in matched_gt_list:
            current_region_mask = np.zeros(padded_gt.shape)
            current_region_mask[gt_mask_labelled == i] = 1
            fn_map = np.maximum(current_region_mask, fn_map)
            
    tp_map = tp_map[(256-64):(64-256),(256-64):(64-256)]
    fp_map = fp_map[(256-64):(64-256),(256-64):(64-256)]
    fn_map = fn_map[(256-64):(64-256),(256-64):(64-256)]
    
    seg_contour_overlay([tp_map,fp_map,fn_map],
                current_img,
                line_size=3,
                fig_size=(16,16),
                thresh=0.5,
                color_list = ['lime','blue','magenta','yellow','red'],
                save_fig=True,
                save_name=contour_save_root + current_photo_name + 'ColorCodedOverlay.jpg',
                show_fig=False,
                title='tp: ' + str(tp) + '\nfp: ' + str(fp) + '\nfn: ' + str(fn),
                save_fig_dpi=150,
                contour_alpha=1)

    return tp, fp, fn

In [16]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

composed = {
    'image':transforms.Compose([ToTensor()]),
    'mask':transforms.Compose([ToTensor()]),
}

In [17]:
data_root = ''

save_root = r'final_E2EC_output_40' + os.sep
os.makedirs(save_root, exist_ok=True)

model_directory = '.' + os.sep + 'saved_models_patch_E2EC' + os.sep
model_list = sorted(glob.glob(model_directory + '*/39.pth'))

full_img_dir = data_root + 'training_images' + os.sep
full_gt_dir = data_root + 'ground_truth' + os.sep

print(len(model_list), model_list[0])

18 ./saved_models_patch_E2EC/1/39.pth


In [18]:
dir_list = sorted(glob.glob(full_img_dir + os.sep + '*'))

img_paths_dict = {}
gt_paths_dict = {}
patient_name_list = []
for dir_no in range(len(dir_list)):
    directory = dir_list[dir_no]
    patient_name = os.path.basename(directory)
    patient_name_list.extend([patient_name])
    
    img_paths_dict[patient_name] = sorted(glob.glob(directory + os.sep + '*.png'))
    gt_paths_dict[patient_name] = sorted(glob.glob(full_gt_dir + patient_name_list[dir_no] + os.sep+ '*.png'))

In [19]:
# list_of_test_patients = list(optimum_iou_thresh_dict.keys())
list_of_test_patients = patient_name_list
patient_dict = {'1': '051301', '2': '051302', '3': '051307', '4': '051308', '5': '051309', '6': '051310',
                '7': '213', '8': '221', '9': '227', '10': '236', '11': '238', '12': '239', 
                '13': '247', '14': '248', '15': 'Kole1', '16': 'Kole2', '17': 'Kole3', '18': 'MPX subject coming from Dekese (Not enrolled)'}

E2EC_metric_arr = {'tp': [], 'fp': [], 'fn': []}
for patient_name in list_of_test_patients:
    model_found = False
    for current_model_name in model_list:
        current_model_name_split = current_model_name.split('/')
        if patient_dict[current_model_name_split[-2]] == patient_name:
            pretrained_model_name = current_model_name
            model_found = True
    
    if model_found:
        cfg = get_cfg()
        pretrained_model = make_network.get_network(cfg).cuda()
        load_network(pretrained_model, pretrained_model_name)
        pretrained_model.eval()
    else:
        print('No model found for patient ' + patient_name)
        break
    
    # Predict on held-out patient
    for img_no in range(len(img_paths_dict[patient_name])): #predict on all photos for current patient
        current_img_path = img_paths_dict[patient_name][img_no]
        current_gt_path = gt_paths_dict[patient_name][img_no]
        
        current_image_name = os.path.basename(current_img_path)[:-4]
        
        print(' Testing image "' + current_image_name + '"')
        
        test_img = np.asarray(cv2.imread(current_img_path))
        test_gt = np.asarray(cv2.imread(current_gt_path))[:,:,0:3]
        test_gt_binary = np.max(test_gt,2).astype('bool')
        
        bbox_list, conf_list, edge_list, mask_list, index_list = patched_inference_list_E2EC(test_img, pretrained_model, patch_size=256, stride=64)
#         indicies, grouping_dict = custom_nms(bbox_list, conf_list, optimum_conf_thresh_dict[patient_name], optimum_iou_thresh_dict[patient_name], edge_list)
        indicies, grouping_dict = custom_nms(bbox_list, conf_list, 0.5, 0.05, edge_list)

        patch_size = 256
        stride = 64
        pad_val = patch_size-stride
        
        bg_masked_indicies = []
        bg_masked_name = current_img_path.replace('training_images', 'images_bgmasked')
        bg_masked_name = bg_masked_name.replace('.png', '_masked.png')
        bg_masked_img = cv2.imread(bg_masked_name)
        bg_masked_img = np.pad(bg_masked_img, pad_width=[(pad_val, pad_val),(pad_val, pad_val),(0, 0)], mode='constant')
        for les_index in indicies:
            if np.sum(bg_masked_img[int(bbox_list[les_index][1]):int(bbox_list[les_index][3]), int(bbox_list[les_index][0]):int(bbox_list[les_index][2])]) > 0:
                bg_masked_indicies.append(les_index)
        
        lesion_count = len(bg_masked_indicies)
        tp, fp, fn = lesion_matching_visual(bg_masked_indicies, test_gt_binary, mask_list, index_list, grouping_dict, current_image_name, test_img)
        E2EC_metric_arr['tp'].extend([tp])
        E2EC_metric_arr['fp'].extend([fp])
        E2EC_metric_arr['fn'].extend([fn])
        
        pr_mask_full = np.zeros(np.pad(test_img, pad_width=[(pad_val, pad_val),(pad_val, pad_val),(0, 0)], mode='constant').shape[0:2])
        count_mask = np.zeros(pr_mask_full.shape)
        for mask_index in bg_masked_indicies:
            for i in range(len(grouping_dict[mask_index])):
                pr_mask_full[int(index_list[grouping_dict[mask_index][i]][0]*stride):int(index_list[grouping_dict[mask_index][i]][0]*stride+patch_size), int(index_list[grouping_dict[mask_index][i]][1]*stride):int(index_list[grouping_dict[mask_index][i]][1]*stride+patch_size)] =  pr_mask_full[int(index_list[grouping_dict[mask_index][i]][0]*stride):int(index_list[grouping_dict[mask_index][i]][0]*stride+patch_size), int(index_list[grouping_dict[mask_index][i]][1]*stride):int(index_list[grouping_dict[mask_index][i]][1]*stride+patch_size)] + mask_list[grouping_dict[mask_index][i]]
                count_mask[int(index_list[grouping_dict[mask_index][i]][0]*stride):int(index_list[grouping_dict[mask_index][i]][0]*stride+patch_size), int(index_list[grouping_dict[mask_index][i]][1]*stride):int(index_list[grouping_dict[mask_index][i]][1]*stride+patch_size)] += (mask_list[grouping_dict[mask_index][i]] > 0)
        count_mask = np.maximum(count_mask, np.ones(pr_mask_full.shape))
        pr_mask_full = pr_mask_full / count_mask
        pr_mask_optimum = pr_mask_full > 0.5
        pr_mask_optimum = pr_mask_optimum[(256-64):(64-256),(256-64):(64-256)]
        pr_mask_full = pr_mask_full[(256-64):(64-256),(256-64):(64-256)]
        
        os.makedirs(save_root + os.sep + patient_name, exist_ok=True)
        save_name = save_root + os.sep + patient_name + os.sep + current_image_name + '_PredictionResults.h5'
        
        with h5py.File(save_name, 'w') as hdf:
            hdf.create_dataset('img', data=test_img, dtype='uint8')
            hdf.create_dataset('gt_mask', data=(test_gt_binary*255).astype('uint8'), dtype='uint8')
            hdf.create_dataset('pr_map', data=(pr_mask_full).astype('double'), dtype='double')
            hdf.create_dataset('pr_mask_optimum', data=(pr_mask_optimum*255).astype('uint8'), dtype='uint8')
            hdf.create_dataset('lesion_count', data=lesion_count, dtype='int')
            # Save E2EC_metric_arr into .npy file
np.save('E2EC_metric_arr.npy', E2EC_metric_arr)

load model: ./saved_models_patch_E2EC/1/39.pth
 Testing image "051301 D2_06"


/home/jiangb/.conda/envs/e2ec/lib/python3.7/site-packages/torch/nn/functional.py:3385: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn("Default grid_sample and affine_grid behavior has changed "


 Testing image "051301 D2_07"
 Testing image "051301 D2_08"
 Testing image "051301 D8_02"
load model: ./saved_models_patch_E2EC/2/39.pth
 Testing image "051302 D2_01"
 Testing image "051302 D2_02"
load model: ./saved_models_patch_E2EC/3/39.pth
 Testing image "051307 010"
 Testing image "051307 012"
load model: ./saved_models_patch_E2EC/4/39.pth
 Testing image "051308 001"
 Testing image "051308 004"
 Testing image "051308 005"
 Testing image "051308 007"
 Testing image "051308 009"
 Testing image "051308 010"
 Testing image "051308 011"
 Testing image "051308 014"
 Testing image "051308 015"
load model: ./saved_models_patch_E2EC/5/39.pth
 Testing image "051309 065"
 Testing image "051309 071"
 Testing image "051309 076"
 Testing image "051309 083"
load model: ./saved_models_patch_E2EC/6/39.pth
 Testing image "051310 D00 001"
 Testing image "051310 D00 005"
 Testing image "051310 D02 008"
 Testing image "051310 D3 001"
 Testing image "051310 D3 002"
 Testing image "051310 D3 003"
load m